# 🤖 Transformer Mimarisi — Sıfırdan PyTorch
# Transformer Architecture — From Scratch with PyTorch

**Yazar / Author:** Merve Caliskan  
**Kaynak / Reference:** *Attention Is All You Need* (Vaswani et al., 2017)

---

## 📋 Notebook Akışı / Flow
| Bölüm | İçerik |
|---|---|
| 1 | Hiperparametreler & Kurulum |
| 2 | Positional Encoding |
| 3 | Multi-Head Attention |
| 4 | Encoder Layer & Full Encoder |
| 5 | Decoder Layer (Masked Attention) |
| 6 | Tam Transformer (Encoder-Decoder) |
| 7 | SST-2 Duygu Analizi — Gerçek Veriyle Eğitim |
| 8 | Transformer LR Scheduler |
| 9 | Mixed Precision Training (FP16) |
| 10 | Beam Search Decoding |

---

> **Bu notebook ne gösteriyor?**  
> GPT ve BERT gibi modellerin temelinde yatan Transformer mimarisini hiçbir kütüphane sihri olmadan,  
> saf PyTorch ile adım adım inşa ediyoruz.
>
> **What does this notebook show?**  
> We build the Transformer architecture that powers GPT and BERT from scratch using pure PyTorch — no magic, just math.

## ⚙️ Bölüm 1: Kurulum & Hiperparametreler
## Section 1: Setup & Hyperparameters

- `d_model` → Her token'ın temsil edildiği vektör boyutu. 512 = orijinal paper değeri.
- `num_heads` → Attention başı sayısı. Her baş farklı ilişkilere odaklanır.
- `d_ff` → Feed-Forward katman genişliği. Genellikle `d_model * 4`.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ── Hiperparametreler / Hyperparameters ──────────────────────
d_model    = 512   # Token embedding boyutu / Token embedding size
num_heads  = 8     # Attention başı sayısı / Number of attention heads
num_layers = 6     # Encoder/Decoder katman sayısı / Number of encoder/decoder layers
d_ff       = 2048  # Feed-forward gizli boyut / Feed-forward hidden size (d_model * 4)
dropout_rate = 0.1 # Dropout oranı / Dropout rate
max_len    = 1000  # Maksimum dizi uzunluğu / Maximum sequence length

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Cihaz / Device: {device}')
print(f'd_model={d_model}, heads={num_heads}, d_ff={d_ff}')

Cihaz / Device: cuda
d_model=512, heads=8, d_ff=2048


## 📍 Bölüm 2: Positional Encoding
## Section 2: Positional Encoding

Transformer'ın sıra bilgisi yoktur — tüm tokenlara aynı anda bakar.  
Positional Encoding her token'a konum bilgisi ekler.

Transformer has no notion of order — it looks at all tokens simultaneously.  
Positional Encoding injects position information into each token.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # PE matrisi: [max_len, d_model]
        # PE matrix: [max_len, d_model]
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()          # [max_len, 1]
        den = torch.exp(torch.arange(0, d_model, 2).float()          # payda / denominator
                        * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * den)   # Çift indeksler → sin / Even indices → sin
        pe[:, 1::2] = torch.cos(pos * den)   # Tek indeksler → cos / Odd indices → cos

        # Buffer olarak kaydet — gradient hesaplanmaz
        # Register as buffer — no gradient computed
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:, :x.size(1)]  # Embedding + pozisyon bilgisi

# Test
pe    = PositionalEncoding(d_model=512)
dummy = torch.zeros(2, 10, 512)  # [batch=2, seq=10, d_model=512]
out   = pe(dummy)
print(f'PE çıktı şekli / PE output shape: {out.shape}')  # [2, 10, 512]

PE çıktı şekli / PE output shape: torch.Size([2, 10, 512])


## 🎯 Bölüm 3: Multi-Head Attention
## Section 3: Multi-Head Attention

Attention mekanizması: her token diğer tüm tokenlara ne kadar dikkat etmeli?

Attention mechanism: how much should each token attend to all other tokens?

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query):** Ben ne arıyorum? / What am I looking for?
- **K (Key):** Ben ne içeriyorum? / What do I contain?
- **V (Value):** Bulunca ne vereyim? / What do I give when found?

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model, num_heads ile bölünebilmeli'

        # PyTorch'un hazır MHA modülü
        # PyTorch's built-in MHA module
        self.attn    = nn.MultiheadAttention(d_model, num_heads,
                                              dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(d_model)  # Residual + LayerNorm
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        # Attention hesapla / Compute attention
        attn_out, _ = self.attn(query, key, value, attn_mask=mask)

        # Residual connection + LayerNorm
        # "Ne öğrendiğini orijinal bilgiye ekle"
        # "Add what you learned to original information"
        return self.norm(query + self.dropout(attn_out))

# Test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x   = torch.rand(2, 10, 512)   # [batch=2, seq=10, d_model=512]
out = mha(x, x, x)             # Self-attention: Q=K=V=x
print(f'MHA çıktı / MHA output: {out.shape}')  # [2, 10, 512]

MHA çıktı / MHA output: torch.Size([2, 10, 512])


## 🔧 Bölüm 4: Encoder Layer & Full Encoder
## Section 4: Encoder Layer & Full Encoder

Bir Encoder katmanı = Self-Attention + Feed-Forward Network (FFN)  
One Encoder layer = Self-Attention + Feed-Forward Network (FFN)

```
Input → [Self-Attention → Add&Norm] → [FFN → Add&Norm] → Output
```

In [4]:
class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),  # Genişlet / Expand
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),  # Daralt / Contract
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x + self.net(x))  # Residual + LayerNorm


class EncoderLayer(nn.Module):
    """Tek bir Encoder katmanı / Single Encoder layer"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn       = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        x = self.self_attn(x, x, x, mask)  # Self-attention
        x = self.ffn(x)                     # Feed-forward
        return x


class Encoder(nn.Module):
    """N adet EncoderLayer'ı üst üste koyar / Stacks N EncoderLayers"""
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)
        self.layers  = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.dropout(self.pos_enc(self.embed(x)))  # Embed + Pos
        for layer in self.layers:
            x = layer(x, mask)                          # N katman / N layers
        return self.norm(x)

# Test
enc = Encoder(vocab_size=1000, d_model=512, num_layers=6,
              num_heads=8, d_ff=2048)
x   = torch.randint(0, 1000, (2, 10))   # [batch=2, seq=10] token IDs
out = enc(x)
print(f'Encoder çıktı / Encoder output: {out.shape}')  # [2, 10, 512]

Encoder çıktı / Encoder output: torch.Size([2, 10, 512])


## 🔓 Bölüm 5: Decoder Layer — Masked Attention
## Section 5: Decoder Layer — Masked Attention

Decoder'ın encoder'dan farkı: **Masked Self-Attention** içerir.  
Decoder's difference from encoder: contains **Masked Self-Attention**.

**Neden mask?** Eğitimde decoder gelecekteki tokenlara bakmamalı.  
**Why mask?** During training, decoder must not see future tokens.

```
"The cat sat" → 'sat' üretirken 'cat' ve 'The' görülebilir, sonrası görülemez.
```

In [5]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # 1. Masked Self-Attention — sadece geçmişe bak / only look at past
        self.masked_attn   = MultiHeadAttention(d_model, num_heads, dropout)
        # 2. Cross-Attention — encoder çıktısına bak / look at encoder output
        self.cross_attn    = MultiHeadAttention(d_model, num_heads, dropout)
        # 3. Feed-Forward
        self.ffn           = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention (hedef diziyi işle)
        #    Masked self-attention (process target sequence)
        x = self.masked_attn(x, x, x, tgt_mask)

        # 2. Cross-attention (encoder'a bak)
        #    Cross-attention (attend to encoder)
        x = self.cross_attn(x, enc_out, enc_out, src_mask)

        # 3. FFN
        return self.ffn(x)


def make_causal_mask(seq_len):
    """Üst üçgen maske — gelecek tokenleri gizler / Upper triangular mask — hides future tokens"""
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return mask  # True olan yerler maskelenir / True positions are masked

# Test
dec_layer = DecoderLayer(d_model=512, num_heads=8, d_ff=2048)
tgt       = torch.rand(2, 8, 512)   # Hedef dizi / Target sequence
enc_out   = torch.rand(2, 10, 512)  # Encoder çıktısı / Encoder output
mask      = make_causal_mask(8)
out       = dec_layer(tgt, enc_out, tgt_mask=mask)
print(f'Decoder Layer çıktı: {out.shape}')  # [2, 8, 512]
print(f'Causal mask (8x8):\n{mask.int()}')

Decoder Layer çıktı: torch.Size([2, 8, 512])
Causal mask (8x8):
tensor([[0, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)


## 🏗️ Bölüm 6: Tam Transformer (Encoder-Decoder)
## Section 6: Full Transformer (Encoder-Decoder)

Tüm parçaları birleştiriyoruz: Encoder + Decoder + çıkış projeksiyonu.  
Putting it all together: Encoder + Decoder + output projection.

In [6]:
class FullTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512,
                 num_layers=6, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()

        # Encoder tarafı / Encoder side
        self.encoder = Encoder(src_vocab, d_model, num_layers,
                                num_heads, d_ff, dropout)

        # Decoder gömme + pozisyon / Decoder embedding + position
        self.dec_embed   = nn.Embedding(tgt_vocab, d_model)
        self.dec_pos     = PositionalEncoding(d_model)
        self.dec_dropout = nn.Dropout(dropout)
        self.dec_layers  = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.dec_norm = nn.LayerNorm(d_model)

        # Çıkış projeksiyonu: d_model → vocab / Output projection: d_model → vocab
        self.output_proj = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Encoder: kaynak diziyi işle / Encode source sequence
        enc_out = self.encoder(src, src_mask)

        # Decoder: hedef diziyi + encoder çıktısını işle
        # Decoder: process target + encoder output
        x = self.dec_dropout(self.dec_pos(self.dec_embed(tgt)))
        for layer in self.dec_layers:
            x = layer(x, enc_out, src_mask, tgt_mask)
        x = self.dec_norm(x)

        # Logit'lere dönüştür / Convert to logits
        return self.output_proj(x)  # [batch, seq, tgt_vocab]


# Test: rastgele ileri geçiş / Test: random forward pass
model = FullTransformer(src_vocab=5000, tgt_vocab=5000,
                         d_model=256, num_layers=2, num_heads=4, d_ff=512)
src   = torch.randint(0, 5000, (2, 10))
tgt   = torch.randint(0, 5000, (2, 8))
tmask = make_causal_mask(8)
out   = model(src, tgt, tgt_mask=tmask)
print(f'Tam Transformer çıktı: {out.shape}')  # [2, 8, 5000]

total = sum(p.numel() for p in model.parameters())
print(f'Toplam parametre / Total parameters: {total/1e6:.1f}M')

Tam Transformer çıktı: torch.Size([2, 8, 5000])
Toplam parametre / Total parameters: 6.5M


## 📊 Bölüm 7: SST-2 Duygu Analizi — Gerçek Eğitim
## Section 7: SST-2 Sentiment Analysis — Real Training

Yukarıda inşa ettiğimiz mimariye benzer bir Transformer sınıflandırıcıyı  
**Stanford Sentiment Treebank (SST-2)** veri seti üzerinde eğitiyoruz.

Training a Transformer classifier on **SST-2** (binary sentiment: positive/negative).

- Dataset: GLUE SST-2 — film yorumlarından pozitif/negatif / movie review sentiment
- Model: Transformer Encoder + sınıflandırma başı / classification head

In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

# Ayarlar / Settings
D_MODEL    = 256
NHEAD      = 4
NUM_LAYERS = 2
DROPOUT    = 0.1
MAX_LEN    = 128
BATCH_SIZE = 32
VOCAB_MODEL = 'bert-base-uncased'  # Tokenizer için / For tokenizer

# Veriyi yükle / Load data
print('SST-2 yükleniyor... / Loading SST-2...')
raw = load_dataset('glue', 'sst2')

# Tokenizer
tokenizer  = AutoTokenizer.from_pretrained(VOCAB_MODEL)
VOCAB_SIZE = tokenizer.vocab_size

def tokenize(examples):
    return tokenizer(examples['sentence'], truncation=True,
                     padding='max_length', max_length=MAX_LEN)

tokenized = raw.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(tokenized['train'], batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=BATCH_SIZE)

print(f'Eğitim / Train: {len(tokenized["train"])} örnek')
print(f'Validasyon / Val: {len(tokenized["validation"])} örnek')

SST-2 yükleniyor... / Loading SST-2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Eğitim / Train: 67349 örnek
Validasyon / Val: 872 örnek


In [8]:
class TransformerClassifier(nn.Module):
    """Transformer Encoder + Sınıflandırma başı / + Classification head"""
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes, dropout):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)  # [CLS] token → sınıf

    def forward(self, input_ids, attention_mask=None):
        x = self.dropout(self.pos_enc(self.embed(input_ids)))

        # Padding maskesi: 0 olan yerleri maskele / Mask padding positions
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.encoder(x, src_key_padding_mask=src_key_padding_mask)

        # [CLS] token (ilk token) sınıflandırma için
        # [CLS] token (first token) for classification
        return self.classifier(x[:, 0, :])


model     = TransformerClassifier(VOCAB_SIZE, D_MODEL, NHEAD,
                                    NUM_LAYERS, 2, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

total = sum(p.numel() for p in model.parameters())
print(f'Model parametreleri / Parameters: {total/1e6:.1f}M')
print(f'Model cihazda / Model on device: {device}')

Model parametreleri / Parameters: 9.4M
Model cihazda / Model on device: cuda


In [9]:
# Tek epoch eğitim döngüsü (demo)
# Single epoch training loop (demo)
# Tam eğitim için num_epochs=5-10 yap / For full training set num_epochs=5-10

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


print('Eğitim başlıyor (1 epoch demo)... / Training (1 epoch demo)...')
loss, acc = train_epoch(model, train_loader, optimizer, criterion)
print(f'Train Loss: {loss:.4f} | Train Acc: {acc:.4f}')

Eğitim başlıyor (1 epoch demo)... / Training (1 epoch demo)...
Train Loss: 0.5898 | Train Acc: 0.6767


## 📈 Bölüm 8: Transformer LR Scheduler
## Section 8: Transformer LR Scheduler

Orijinal paper'daki öğrenme oranı programı: warmup + decay.  
Learning rate schedule from original paper: warmup + decay.

$$lr = d_{model}^{-0.5} \cdot \min(step^{-0.5},\ step \cdot warmup^{-1.5})$$

In [10]:
import torch.optim as optim

class TransformerLRScheduler(optim.lr_scheduler._LRScheduler):
    """
    Orijinal Transformer paper'ındaki LR programı.
    LR schedule from original Transformer paper.
    """
    def __init__(self, optimizer, d_model, warmup_steps=4000, last_epoch=-1):
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = max(1, self._step_count)
        # Warmup: lineer artış / warmup: linear increase
        # Sonra: 1/sqrt(step) azalış / After: 1/sqrt(step) decay
        scale = (self.d_model ** -0.5) * min(
            step ** -0.5,
            step * (self.warmup_steps ** -1.5)
        )
        return [base_lr * scale for base_lr in self.base_lrs]


# LR değişimini görselleştir / Visualize LR change
opt  = torch.optim.Adam(model.parameters(), lr=1.0)
sch  = TransformerLRScheduler(opt, d_model=D_MODEL, warmup_steps=4000)

lrs  = []
for _ in range(10000):
    lrs.append(sch.get_lr()[0])
    sch.step()

peak_step = lrs.index(max(lrs))
print(f'Peak LR adımı / Peak LR step: {peak_step}')
print(f'Peak LR değeri / Peak LR value: {max(lrs):.6f}')
print(f'Warmup bölgesi: 0-{peak_step} | Decay bölgesi: {peak_step}+')

Peak LR adımı / Peak LR step: 3999
Peak LR değeri / Peak LR value: 0.000988
Warmup bölgesi: 0-3999 | Decay bölgesi: 3999+


/tmp/ipykernel_662/847889170.py:31: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sch.step()


## ⚡ Bölüm 9: Mixed Precision Training (FP16)
## Section 9: Mixed Precision Training (FP16)

FP16 (16-bit float) kullanarak:
- 2x daha hızlı eğitim / 2x faster training
- ~%40 daha az GPU RAM / ~40% less GPU RAM
- Neredeyse aynı doğruluk / Nearly same accuracy

**GradScaler:** FP16'da gradyan taşmasını önler / Prevents gradient overflow in FP16

In [11]:
from torch.cuda.amp import autocast, GradScaler

scaler    = GradScaler()  # FP16 gradyan ölçekleyici / FP16 gradient scaler
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

def train_step_amp(model, batch, optimizer, criterion, scaler):
    """Mixed precision eğitim adımı / Mixed precision training step"""
    ids    = batch['input_ids'].to(device)
    mask   = batch['attention_mask'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()

    # autocast: forward pass FP16 ile / forward pass with FP16
    with autocast():
        logits = model(ids, mask)
        loss   = criterion(logits, labels)

    # scaler: gradyanları ölçekle, güncelle / scale gradients, update
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    return loss.item()


# İlk batch ile test / Test with first batch
first_batch = next(iter(train_loader))
loss = train_step_amp(model, first_batch, optimizer, criterion, scaler)
print(f'AMP eğitim adımı tamamlandı / AMP training step complete')
print(f'Loss: {loss:.4f}')
print(f'GradScaler scale: {scaler.get_scale()}')

/tmp/ipykernel_662/1686574471.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = GradScaler()  # FP16 gradyan ölçekleyici / FP16 gradient scaler
/tmp/ipykernel_662/1686574471.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


AMP eğitim adımı tamamlandı / AMP training step complete
Loss: 0.4432
GradScaler scale: 65536.0


## 🔍 Bölüm 10: Beam Search Decoding
## Section 10: Beam Search Decoding

Greedy search: her adımda en yüksek olasılıklı tokeni seç → kısa vadeli optimal.  
Greedy search: pick highest probability token each step → locally optimal.

**Beam Search:** En iyi `beam_size` hipotezi aynı anda takip et → daha iyi sonuç.  
**Beam Search:** Track top `beam_size` hypotheses simultaneously → better results.

```
Adım 1: ["The"]        → score=-0.3
Adım 2: ["The cat"]    → score=-0.7  ← beam 1
         ["The dog"]   → score=-0.8  ← beam 2
Adım 3: ["The cat sat"] → score=-1.1  ← kazanan / winner
```

In [12]:
BEAM_SIZE   = 3    # Kaç hipotez takip edilsin / How many hypotheses to track
MAX_LEN_TGT = 50   # Maksimum çıktı uzunluğu / Max output length
PAD_IDX     = 0
EOS_IDX     = 2    # Cümle sonu tokeni / End of sentence token


def beam_search(model, src, beam_size=3, max_len=50):
    """
    Basit beam search implementasyonu.
    Simple beam search implementation.

    Her hipotez: (log_olasılık, token_listesi)
    Each hypothesis: (log_probability, token_list)
    """
    model.eval()
    with torch.no_grad():
        # Encoder'ı çalıştır / Run encoder
        enc_out = model.encoder(src)

        # Başlangıç: BOS (beginning of sequence) tokeni
        # Start: BOS (beginning of sequence) token
        beams = [(0.0, [1])]  # (log_prob, tokens), 1 = BOS token

        for step in range(max_len):
            candidates = []

            for log_prob, tokens in beams:
                # EOS geldiyse bu beam tamamlandı / If EOS, this beam is complete
                if tokens[-1] == EOS_IDX:
                    candidates.append((log_prob, tokens))
                    continue

                # Mevcut token dizisini decoder'a ver
                # Feed current token sequence to decoder
                tgt    = torch.tensor([tokens]).long()
                tmask  = make_causal_mask(len(tokens))

                # Logit'leri al / Get logits
                dec_in = model.dec_dropout(model.dec_pos(model.dec_embed(tgt)))
                for layer in model.dec_layers:
                    dec_in = layer(dec_in, enc_out, tgt_mask=tmask)
                logits = model.output_proj(dec_in[:, -1, :])  # Son token

                # Top-k token seç / Select top-k tokens
                log_probs = F.log_softmax(logits, dim=-1)
                topk_probs, topk_ids = log_probs.topk(beam_size)

                for i in range(beam_size):
                    new_prob   = log_prob + topk_probs[0, i].item()
                    new_tokens = tokens + [topk_ids[0, i].item()]
                    candidates.append((new_prob, new_tokens))

            # En iyi beam_size adayı tut / Keep best beam_size candidates
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]

            # Hepsi EOS'a ulaştıysa dur / Stop if all reached EOS
            if all(b[1][-1] == EOS_IDX for b in beams):
                break

    # En yüksek skorlu hipotezi döndür / Return highest scoring hypothesis
    return beams[0][1]


# Demo: FullTransformer model ile beam search
# Demo: beam search with FullTransformer model
demo_model = FullTransformer(src_vocab=5000, tgt_vocab=5000,
                               d_model=128, num_layers=2, num_heads=4, d_ff=256)
src_demo   = torch.randint(0, 5000, (1, 10))  # [batch=1, seq=10]
result     = beam_search(demo_model, src_demo, beam_size=BEAM_SIZE)

print(f'Beam Search tamamlandı / Beam Search complete')
print(f'Beam size: {BEAM_SIZE}')
print(f'Üretilen token IDs / Generated token IDs: {result}')
print(f'Uzunluk / Length: {len(result)} token')

Beam Search tamamlandı / Beam Search complete
Beam size: 3
Üretilen token IDs / Generated token IDs: [1, 4720, 2029, 931, 3113, 1786, 1977, 2725, 2572, 1088, 3657, 728, 4170, 4495, 2930, 3601, 1412, 3649, 4698, 1252, 3101, 4175, 1162, 4281, 1447, 714, 341, 1445, 362, 1185, 1926, 1352, 1134, 4538, 855, 4174, 3166, 696, 1185, 1926, 2530, 3165, 3, 4264, 4128, 3729, 4379, 1445, 362, 2256, 1157]
Uzunluk / Length: 51 token


## 🎓 Özet / Summary

Bu notebook'ta sıfırdan inşa ettiğimiz parçalar:
What we built from scratch in this notebook:

| Bileşen | Açıklama |
|---|---|
| Positional Encoding | Sin/cos ile konum bilgisi |
| Multi-Head Attention | Q, K, V projeksiyonları + scaled dot-product |
| Feed-Forward Network | 2 katmanlı MLP, residual + LayerNorm |
| Encoder | N × EncoderLayer |
| Decoder | N × DecoderLayer (masked + cross attention) |
| Full Transformer | Encoder-Decoder + output projection |
| Transformer LR | Warmup + inverse sqrt decay |
| Mixed Precision | FP16 + GradScaler |
| Beam Search | Top-k hipotez takibi |
| SST-2 Fine-tuning | Gerçek veriyle eğitim |

